In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# --- Path resolution (identical convention to the other two notebooks) ---
_cwd = Path.cwd()
if (_cwd / "data" / "raw" / "DHS_Districts").is_dir():
    REPO_ROOT = _cwd
elif (_cwd.parent / "data" / "raw" / "DHS_Districts").is_dir():
    REPO_ROOT = _cwd.parent
else:
    raise FileNotFoundError(f"Could not find data/raw/DHS_Districts from cwd {_cwd}.")

PROCESSED_DIR = REPO_ROOT / "data" / "processed"

tfr_path = PROCESSED_DIR / "tfr_panel_full.parquet"
intensity_path = PROCESSED_DIR / "district_intensity_variables.parquet"

for p in [tfr_path, intensity_path]:
    print(p, "->", p.exists())

tfr_panel = pd.read_parquet(tfr_path)
district_intensity = pd.read_parquet(intensity_path)

print("TFR panel shape:", tfr_panel.shape)
print("District intensity shape:", district_intensity.shape)

# --- Merge on stable_district_id ---
merged = tfr_panel.merge(district_intensity, on="stable_district_id", how="left", suffixes=("", "_intensity"))
print("Merged shape (pre-restriction):", merged.shape)

n_missing_intensity = merged["delta_mobile_ownership"].isna().sum()
print(f"Rows with no matching intensity data: {n_missing_intensity} / {len(merged)}")

# --- Approximate per-district standard error for delta_mobile_ownership ---
# SE(delta) = sqrt(Var(p4) + Var(p5)), each Var(p) = p(1-p)/n -- standard
# binomial-proportion SE, combined in quadrature since the two rounds are
# independent samples
p4, n4 = merged["s930_mean"], merged["n_resp_nfhs4"]
p5, n5 = merged["v169a_mean"], merged["n_resp_nfhs5"]
merged["delta_mobile_ownership_se"] = np.sqrt((p4 * (1 - p4) / n4) + (p5 * (1 - p5) / n5))

# --- Restrict to the pre-period, rateable windows only ---
# year_approx <= 2015 (strictly pre-Jio; 2016 is ambiguous given only-annual
# granularity and Jio's Sept 2016 launch, so excluded from the clean pre-period)
RATEABLE_WINDOWS = {"primary", "sensitivity"}
pre_period = merged[
    merged["window"].isin(RATEABLE_WINDOWS) & (merged["year_approx"] <= 2015)
].copy()

print("\nPre-period (rateable, year_approx<=2015) shape:", pre_period.shape)
print("Year range:", pre_period["year_approx"].min(), "-", pre_period["year_approx"].max())
print("\nRows by round:")
print(pre_period["round"].value_counts())
print("\nDistricts covered:", pre_period["stable_district_id"].nunique(), "/ 575")
print("\nTercile distribution (delta_mobile_ownership_tercile):")
print(pre_period["delta_mobile_ownership_tercile"].value_counts())

pre_period.sort_values(["stable_district_id", "round", "year_approx"]).head(10)

/Users/eknoorsandhu/DHS_India_Research/data/processed/tfr_panel_full.parquet -> True
/Users/eknoorsandhu/DHS_India_Research/data/processed/district_intensity_variables.parquet -> True
TFR panel shape: (38664, 13)
District intensity shape: (575, 15)
Merged shape (pre-restriction): (38664, 27)
Rows with no matching intensity data: 0 / 38664

Pre-period (rateable, year_approx<=2015) shape: (14950, 28)
Year range: 2001.0 - 2015.0

Rows by round:
round
NFHS-4    8625
NFHS-5    6325
Name: count, dtype: int64

Districts covered: 575 / 575

Tercile distribution (delta_mobile_ownership_tercile):
delta_mobile_ownership_tercile
low     4992
high    4992
mid     4966
Name: count, dtype: int64


,round,stable_district_id,years_before_survey,window,TFR,total_weighted_births,total_women_years,n_age_bands_present,year_approx,low_cell_count,...,delta_mobile_ownership,nfhs5_mobile_level,nfhs5_internet_level,low_n_either_round,delta_mobile_ownership_tercile,nfhs5_mobile_level_tercile,nfhs5_internet_level_tercile,district_name_nfhs4_intensity,district_name_nfhs5_intensity,delta_mobile_ownership_se
15,NFHS-4,1.0,15.0,sensitivity,3.355340,22.063440,144.230683,4.0,2001.0,True,...,0.22746,0.709814,0.454159,False,high,high,high,kupwara,kupwara,0.044163
14,NFHS-4,1.0,14.0,sensitivity,4.527983,27.160918,149.586274,5.0,2002.0,True,...,0.22746,0.709814,0.454159,False,high,high,high,kupwara,kupwara,0.044163
13,NFHS-4,1.0,13.0,sensitivity,5.504054,32.136069,166.840930,5.0,2003.0,False,...,0.22746,0.709814,0.454159,False,high,high,high,kupwara,kupwara,0.044163
12,NFHS-4,1.0,12.0,sensitivity,3.851308,27.998202,177.521003,5.0,2004.0,True,...,0.22746,0.709814,0.454159,False,high,high,high,kupwara,kupwara,0.044163
11,NFHS-4,1.0,11.0,sensitivity,4.072479,30.981737,191.401163,5.0,2005.0,False,...,0.22746,0.709814,0.454159,False,high,high,high,kupwara,kupwara,0.044163
10,NFHS-4,1.0,10.0,sensitivity,3.588075,28.515665,203.024481,5.0,2006.0,True,...,0.22746,0.709814,0.454159,False,high,high,high,kupwara,kupwara,0.044163
9,NFHS-4,1.0,9.0,sensitivity,4.650935,31.228200,212.746953,6.0,2007.0,False,...,0.22746,0.709814,0.454159,False,high,high,high,kupwara,kupwara,0.044163
8,NFHS-4,1.0,8.0,sensitivity,4.108681,31.946656,223.684832,6.0,2008.0,False,...,0.22746,0.709814,0.454159,False,high,high,high,kupwara,kupwara,0.044163
7,NFHS-4,1.0,7.0,primary,3.684889,31.305178,240.393251,6.0,2009.0,False,...,0.22746,0.709814,0.454159,False,high,high,high,kupwara,kupwara,0.044163
6,NFHS-4,1.0,6.0,primary,3.707135,32.789613,253.139986,6.0,2010.0,False,...,0.22746,0.709814,0.454159,False,high,high,high,kupwara,kupwara,0.044163
